## Example 15 - Basic locking example

#### Create an environment named: "gdal"

Install Anaconda3-2020.07-Windows-x86_64.exe

In the Anaconda prompt execute the following commands:

    conda create --name gdal
    conda activate gdal

    conda config --add channels conda-forge 
    conda config --set channel_priority strict

    conda install gdal
    yes
    conda install geopandas
    yes
    conda install folium
    yes
    conda install pyyaml
    yes

    cd c:\Users\MRV\OneDrive - Van Oord\Software\github\OpenTNSim\
    pip install -e .

    conda install jupyter
    yes

    cd ..
    jupyter notebook

#### Import packages

In [1]:
# package(s) related to time, space and id
import datetime, time
import os
import io
import functools
import logging
import pickle

# package(s) related to the simulation
import simpy
import networkx as nx  
import numpy as np
import pandas as pd

# OpenTNSim
from opentnsim import core
from opentnsim import plot

# spatial libraries 
import shapely.geometry
import shapely.wkt
import pyproj
import shapely.geometry
import folium

# package(s) for data handling
import requests

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()

# define the coorinate system
geod = pyproj.Geod(ellps="WGS84")

#### Load methods

In [2]:
@functools.lru_cache
def load_fis_network(url):
    """load the topological fairway information system network (vaarweginformatie.nl)"""

    # get the data from the url
    resp = requests.get(url)
    # convert to file object
    stream = io.StringIO(resp.text)
    
    # This will take a minute or two
    # Here we convert the network to a networkx object
    G = nx.read_yaml(stream)

    # some brief info
    n_bytes = len(resp.content)
    msg = '''Loaded network from {url} file size {mb:.2f}MB. Network has {n_nodes} nodes and {n_edges} edges.'''
    summary = msg.format(url=url, mb=n_bytes / 1000**2, n_edges=len(G.edges), n_nodes=len(G.nodes))
    logger.info(summary)

    # The topological network contains information about the original geometry. 
    # Let's convert those into python shapely objects for easier use later
    for n in G.nodes:
        G.nodes[n]['geometry'] = shapely.geometry.Point(G.nodes[n]['X'], G.nodes[n]['Y'])
    for e in G.edges:
        edge = G.edges[e]
        edge['geometry'] = shapely.wkt.loads(edge['Wkt'])
        edge['length'] = edge_length(edge)    
    
    return G 

In [3]:
def find_closest_node(G, point):
    """find the closest node on the graph from a given point"""
    
    distance = np.full((len(G.nodes)), fill_value=np.nan)
    for ii, n in enumerate(G.nodes):
        distance[ii] = point.distance(G.nodes[n]['geometry'])
    name_node = list(G.nodes)[np.argmin(distance)]
    distance_node = np.min(distance)
    
    return name_node, distance_node

In [4]:
def find_closest_edge(G, point):
    """find the closest edge on the graph from a given point"""
    
    distance = np.full((len(G.edges)), fill_value=np.nan)
    for ii, e in enumerate(G.edges):
        distance[ii] = point.distance(G.edges[e]['geometry'])
    name_edge = list(G.edges)[np.argmin(distance)]
    distance_edge = np.min(distance)
    
    return name_edge, distance_edge

In [5]:
def edge_length(edge):
    """compute the great circle length of an edge
    The network version 0.1 contains the lat/lon distance in a length property. 
    But we need the "great circle" or projected distance. 
    Let's define a function to recompute it.
    """
    
    # get the geometry
    geom = edge['geometry']
    # get lon, lat
    lats, lons = np.array(geom).T
    # this requires pyproj 2.3.0
    distance = geod.line_length(lons, lats)

    return distance

#### Load transport network (based on vaarweginformatie.nl)

In [6]:
# Link to the latest version of the 'Vaarweginformatie.nl' network
url = 'https://zenodo.org/record/3981105/files/network_digital_twin_v0.1.yaml'

In [7]:
# create a cached version to speed up loading (remove cached file if a better yaml file is available)
fname = "fis_cache\\{}.pkl".format('FIS')

if os.path.exists(fname):
    print('I am loading cached network')
    with open(fname, 'rb') as pkl_file:
        FG = pickle.load(pkl_file)
        pkl_file.close()

else:
    print('I am getting new network')
    FG = load_fis_network(url)

    os.makedirs(os.path.dirname(fname), exist_ok=True)
    with open(fname, 'wb') as pkl_file:
        pickle.dump(FG, pkl_file)
        pkl_file.close()

I am loading cached network


#### Select area of interest

In [8]:
#This polygon describes the area of the AIS data received from RWS

# North-East
NE = (3.90838623046875, 51.18278643144626)
# South-East
SE = (4.73236083984375, 51.18278643144626)
# South-West
SW = (4.73236083984375, 51.95611479866179)
# North-West
NW = (3.90838623046875, 51.95611479866179)

polygon = shapely.geometry.Polygon([NE, SE, SW, NW])

#### Add the relevant information to the nodes and edges

In [9]:
nodes = []
edges = []

node_names = []

for edge in FG.edges(data = True):
    node_1 = FG.nodes[edge[0]]
    node_2 = FG.nodes[edge[1]]
    
    if node_1["geometry"].within(polygon) or node_2["geometry"].within(polygon):
        nodes.append(node_1)
        nodes.append(node_2)
        node_names.append(edge[0])
        node_names.append(edge[1])
        edges.append(edge)

In [10]:
FG = nx.DiGraph()

for i in range(len(nodes)):
    node = nodes[i]
    node_name = node_names[i]
    FG.add_node(node_name, name = str((node['X'], node['Y'])), Position = (node['X'], node['Y']), geometry = node["geometry"])

for edge in edges:
    FG.add_edge(edge[0], edge[1], Info = edge[2]) 

In [11]:
# The read_shp creates a directed graph with single edges, we require a directed graph but two-way traffic
FG = FG.to_undirected() 
FG = FG.to_directed() 

In [12]:
# create a library of some interesting locations (lon, lat)
locations = {
    'Transferium Maasvlakte': shapely.geometry.Point(4.087406, 51.936737),
    'Neusse': shapely.geometry.Point(6.708892, 51.215737),
    'Basel': shapely.geometry.Point(7.640572, 47.555449),
    'Volkerak locks':  shapely.geometry.Point(4.408202, 51.689836),
    'Before volkerak': shapely.geometry.Point(4.430289, 51.700046),
    'After volkerak': shapely.geometry.Point(4.392555, 51.681250),
    'Kreekrak locks':  shapely.geometry.Point(4.230610, 51.447945),
    'Hansweert locks': shapely.geometry.Point(4.008972, 51.458791),
    'Schelde-Rijn canal-1': shapely.geometry.Point(4.201416987493968, 51.58410292465058), 
    'Schelde-Rijn canal-2': shapely.geometry.Point( 4.22456143757696, 51.5221824639149),
    'Maassluis': shapely.geometry.Point(4.22590208053588, 51.9284782409667),
    'Antwerpen': shapely.geometry.Point(4.291013, 51.373772),
    'Before Kreekrak': shapely.geometry.Point(4.220, 51.50),
    'After Kreekrak': shapely.geometry.Point(4.233996, 51.429589)
}

locations_nodes = {n: find_closest_node(FG, locations[n])[0] for n in locations} #FG_new
locations_nodes

{'Transferium Maasvlakte': 22161408.0,
 'Neusse': 'L1360434_B',
 'Basel': 'L1360434_B',
 'Volkerak locks': 'L42863_B',
 'Before volkerak': 8862498.0,
 'After volkerak': 8868426.0,
 'Kreekrak locks': 'L9305_B',
 'Hansweert locks': 'B49727_A',
 'Schelde-Rijn canal-1': 'B37463_B',
 'Schelde-Rijn canal-2': 8867083.0,
 'Maassluis': 8867449.0,
 'Antwerpen': 'B41483_A',
 'Before Kreekrak': 8868095.0,
 'After Kreekrak': 'B20349_A'}

#### Plot the entire network

In [13]:
# Plot the network and highlight the Volkerak locks
m = folium.Map(location=[51.7, 4.4], zoom_start = 12, tiles="cartodbpositron")

for edge in FG.edges(data = True):
    points_x = list(edge[2]["Info"]["geometry"].coords.xy[0])
    points_y = list(edge[2]["Info"]["geometry"].coords.xy[1])
    
    line = []
    for i, _ in enumerate(points_x):
        line.append((points_y[i], points_x[i]))
        
        
    if edge[2]["Info"]["StartJunctionId"] == 'L42863_A':
        folium.PolyLine(line, color = "red", weight = 5, popup = edge[2]["Info"]["StartJunctionId"]).add_to(m)    
            
    else:
        folium.PolyLine(line, weight = 2, popup = edge[2]["Info"]["StartJunctionId"]).add_to(m)

m

#### Create vessel object

In [14]:
# Make a class out of mix-ins (currently very simple (identifiable and movable only))
TransportResource = type('TransportResource', 
                         (core.Identifiable,   # name, id
                          core.Movable), {})   # v, geometry, route (Locatable, Routable, Log)

In [15]:
fleet = {
            0: {'ship_type': 'Container1',    'start_point': 'Maassluis', 'end_point': 'Antwerpen'},
            1: {'ship_type': 'Container1',    'start_point': 'Before volkerak', 'end_point': 'After volkerak'}
        }


In [16]:
# Initiate vessels
vessels = []
for ship in fleet:
    
    # Start point, end point and route
    start_point = locations_nodes[fleet[ship]['start_point']] 
    end_point = locations_nodes[fleet[ship]['end_point']]   
    path = nx.dijkstra_path(FG, start_point, end_point, weight='length') 
    
    # prepare data for the vessel
    data_vessel = {"env": None,
                   "name": fleet[ship]['ship_type'] ,
                   "route": path,
                   "geometry": FG.nodes[start_point]['geometry'],  # lon, lat
                   "v": 1  # 1 m/s
                  }

    # create the transport processing resource
    vessels.append(TransportResource(**data_vessel))

#### Start simulation

In [17]:
# Start simpy environment (at a given date and time)
simulation_start = datetime.datetime(2020, 9, 26)
env = simpy.Environment(initial_time = time.mktime(simulation_start.timetuple()))
env.epoch = time.mktime(simulation_start.timetuple())

In [18]:
# Add graph to environment
env.FG = FG

#### Rozenburgse lock

In [19]:
rozenburgse_lock = core.IsLock(env = env, nr_resources = 1, priority = True, name = "Rozenburgse lock", 
                        node_1 = "B7951_B", node_2 = "L4199_A",
                        lock_length = 300, lock_width = 24, lock_depth = 4.5, 
                        doors_open = 10 * 60, doors_close = 10 * 60, operating_time = 25 * 60)

In [20]:
for edge in FG.edges(data = True):
    if edge[2]["Info"]["StartJunctionId"] == 'B7951_B':
        # For testing, all locks have the water level at the right side
        rozenburgse_lock.water_level = "B7951_B"
        # Add locks to the correct edge
        FG.edges[edge[0], edge[1]]["Lock"] = [rozenburgse_lock]

#### Volkerak locks (1, 2 and 3)

In [21]:
volkerak_lock_nr_1 = core.IsLock(env = env, nr_resources = 1, priority = True, name = "Volkerak - 1", 
                        node_1 = "L42863_A", node_2 = "L42863_B",
                        lock_length = 330, lock_width = 24, lock_depth = 4.5, 
                        doors_open = 10 * 60, doors_close = 10 * 60, operating_time = 25 * 60)

volkerak_lock_nr_2 = core.IsLock(env = env, nr_resources = 1, priority = True, name = "Volkerak - 2", 
                        node_1 = "L42863_A", node_2 = "L42863_B",
                        lock_length = 330, lock_width = 24, lock_depth = 4.5, 
                        doors_open = 10 * 60, doors_close = 10 * 60, operating_time = 25 * 60)

volkerak_lock_nr_3 = core.IsLock(env = env, nr_resources = 1, priority = True, name = "Volkerak - 3", 
                        node_1 = "L42863_A", node_2 = "L42863_B",
                        lock_length = 330, lock_width = 24, lock_depth = 4.5, 
                        doors_open = 10 * 60, doors_close = 10 * 60, operating_time = 25 * 60)

In [22]:
for edge in FG.edges(data = True): #FG_new
    if edge[2]['Info']['StartJunctionId'] == 'L42863_A':
        # For testing, all locks have the water level at the right side
        volkerak_lock_nr_1.water_level = "L42863_A"
        volkerak_lock_nr_2.water_level = "L42863_A"
        volkerak_lock_nr_3.water_level = "L42863_A"
        
        # Add locks to the correct edge
        FG.edges[edge[0], edge[1]]["Lock"] = [volkerak_lock_nr_1, volkerak_lock_nr_2, volkerak_lock_nr_3] #FG_new

#### Krammer locks (1 and 2)

In [23]:
krammer_lock_nr_1 = core.IsLock(env = env, nr_resources = 1, priority = True, name = "Krammer lock - 1", 
                        node_1 = "L39446_A", node_2 = "L39446_B",
                        lock_length = 280, lock_width = 24, lock_depth = 4.5, 
                        doors_open = 10 * 60, doors_close = 10 * 60, operating_time = 25 * 60)

krammer_lock_nr_2 = core.IsLock(env = env, nr_resources = 1, priority = True, name = "Krammer lock - 2", 
                        node_1 = "L39446_A", node_2 = "L39446_B",
                        lock_length = 280, lock_width = 24, lock_depth = 4.5, 
                        doors_open = 10 * 60, doors_close = 10 * 60, operating_time = 25 * 60)

In [24]:
for edge in FG.edges(data = True):
    if edge[2]['Info']["StartJunctionId"] == 'L39446_A':
        # For testing, all locks have the water level at the right side
        krammer_lock_nr_1.water_level = "L39446_A"
        krammer_lock_nr_2.water_level = "L39446_A"
        
        # Add locks to the correct edge
        FG.edges[edge[0], edge[1]]["Lock"] = [krammer_lock_nr_1, krammer_lock_nr_2]

#### Hansweert locks (1 and 2)

In [25]:
hansweert_lock_nr_1 = core.IsLock(env = env, nr_resources = 1, priority = True, name = "Hansweert lock - 1", 
                        node_1 = "L25235_A", node_2 = "L25235_B",
                        lock_length = 280, lock_width = 24, lock_depth = 4.5, 
                        doors_open = 10 * 60, doors_close = 10 * 60, operating_time = 25 * 60)

hansweert_lock_nr_2 = core.IsLock(env = env, nr_resources = 1, priority = True, name = "Hansweert lock - 2", 
                        node_1 = "L25235_A", node_2 = "L25235_B",
                        lock_length = 280, lock_width = 24, lock_depth = 4.5, 
                        doors_open = 10 * 60, doors_close = 10 * 60, operating_time = 25 * 60)

In [26]:
for edge in FG.edges(data = True):
    if edge[2]['Info']["StartJunctionId"] == 'L25235_A':
        # For testing, all locks have the water level at the right side
        hansweert_lock_nr_1.water_level = "L25235_A"
        hansweert_lock_nr_2.water_level = "L25235_A"
        
        # Add locks to the correct edge
        FG.edges[edge[0], edge[1]]["Lock"] = [hansweert_lock_nr_1, hansweert_lock_nr_2]

#### Kreekrak locks (1 and 2)

In [27]:
kreekrak_lock_nr_1 = core.IsLock(env = env, nr_resources = 1, priority = True, name = "Kreekrak lock - 1", 
                        node_1 = "L9305_A", node_2 = "L9305_B",
                        lock_length = 320, lock_width = 24, lock_depth = 4.5, 
                        doors_open = 10 * 60, doors_close = 10 * 60, operating_time = 25 * 60)

kreekrak_lock_nr_2 = core.IsLock(env = env, nr_resources = 1, priority = True, name = "Kreekrak lock - 2", 
                        node_1 = "L9305_A", node_2 = "L9305_B",
                        lock_length = 320, lock_width = 24, lock_depth = 4.5, 
                        doors_open = 10 * 60, doors_close = 10 * 60, operating_time = 25 * 60)

In [28]:
for edge in FG.edges(data = True):
    if edge[2]['Info']["StartJunctionId"] == 'L9305_A':
        # For testing, all locks have the water level at the right side
        kreekrak_lock_nr_1.water_level = "L9305_A"
        kreekrak_lock_nr_2.water_level = "L9305_A"
        
        # Add locks to the correct edge
        FG.edges[edge[0], edge[1]]["Lock"] = [kreekrak_lock_nr_1, kreekrak_lock_nr_2]

#### Run simulation

In [29]:
# Define the vessel start process
def start(env, vessel):
    """The start process moves a vessel over the network 
    (from its 'vessel.geometry' position to the end of 'vessel.route')"""
    
    vessel.log_entry("Start sailing", env.now, "", vessel.geometry)
    yield from vessel.move()
    vessel.log_entry("Stop sailing", env.now, "", vessel.geometry)

In [30]:
# Register start processes per vessel and run SimPy
for vessel in vessels:
    vessel.env = env
    env.process(start(env, vessel))

env.run()

#### 3.9 Post processing log results

In [32]:
# Visualise vessel movements on network
plot.vessel_kml(env, vessels, fname='vessel_movements.kml')

In [33]:
print(len(vessels[0].log["Message"]))
print(len(vessels[0].log["Timestamp"]))
print(len(vessels[0].log["Value"]))
print(len(vessels[0].log["Geometry"]))

178
178
178
178


In [35]:
df = pd.DataFrame.from_dict(vessels[1].log)
df

,Message,Timestamp,Value,Geometry
0,Start sailing,2020-09-26 00:00:00.000000,,POINT (4.43028931681445 51.700046323868)
1,Sailing from node 8862498.0 to node L42863_A s...,2020-09-26 00:00:00.000000,0,POINT (4.43028931681445 51.700046323868)
2,Sailing from node 8862498.0 to node L42863_A stop,2020-09-26 00:31:15.570119,0,POINT (4.409136190501533 51.6894936665846)
3,Passing lock start,2020-09-26 00:31:15.570119,0,POINT (4.409136190501533 51.6894936665846)
4,Passing lock stop,2020-09-26 01:16:15.570119,2700,POINT (4.40878376619477 51.6893044653478)
5,Sailing from node L42863_B to node B34113_A start,2020-09-26 01:16:15.570119,0,POINT (4.40878376619477 51.6893044653478)
6,Sailing from node L42863_B to node B34113_A stop,2020-09-26 01:17:53.558818,0,POINT (4.407711423268813 51.68872877120878)
7,Sailing from node B34113_A to node B34113_B start,2020-09-26 01:17:53.558818,0,POINT (4.407711423268813 51.68872877120878)
8,Sailing from node B34113_A to node B34113_B stop,2020-09-26 01:18:01.609818,0,POINT (4.407623317192122 51.68868147089958)
9,Sailing from node B34113_B to node 8868426.0 s...,2020-09-26 01:18:01.609818,0,POINT (4.407623317192122 51.68868147089958)
